In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

LANDING_PATH = Path(
    "/content/drive/MyDrive/cloud-provider-analytics-data/landing"
)

BRONZE_PATH = Path(
    "/content/drive/MyDrive/cloud-provider-analytics-data/bronze"
)

print("Landing:", LANDING_PATH)
print("Bronze:", BRONZE_PATH)
print("Existe Landing:", LANDING_PATH.exists())


Mounted at /content/drive
Landing: /content/drive/MyDrive/cloud-provider-analytics-data/landing
Bronze: /content/drive/MyDrive/cloud-provider-analytics-data/bronze
Existe Landing: True


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CloudProviderAnalytics_BatchBronze")
    .master("local[*]")
    .getOrCreate()
)

print("Versión de Spark:", spark.version)


Versión de Spark: 4.0.5


In [ ]:
archivos = [
    "customers_orgs.csv",
    "users.csv",
    "billing_monthly.csv"
]

for archivo in archivos:
    ruta = LANDING_PATH / archivo
    print(f"{archivo}: {'OK' if ruta.exists() else 'NO ENCONTRADO'}")


customers_orgs.csv: OK
users.csv: OK
billing_monthly.csv: OK


In [ ]:
# Leemos el archivo como texto, sin inferir tipos automáticamente
customers_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(str(LANDING_PATH / "customers_orgs.csv"))
)

print("Cantidad de registros:", customers_raw.count())

print("\nColumnas disponibles:")
print(customers_raw.columns)

print("\nPrimeros registros:")
customers_raw.show(5, truncate=False)


Cantidad de registros: 80

Columnas disponibles:
['org_id', 'org_name', 'industry', 'hq_region', 'plan_tier', 'is_enterprise', 'signup_date', 'sales_rep', 'lifecycle_stage', 'marketing_source', 'nps_score']

Primeros registros:
+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_id      |org_name       |industry     |hq_region |plan_tier|is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|
+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_xaji0y6d|Nimbus Labs 0  |Education    |sa-east   |standard |True         |2025-05-26 |rep_c    |churned        |partner         |NULL     |
|org_pbhsahxt|Nova Tech 1    |Education    |us-east   |standard |False        |2025-06-16 |rep_d    |lead           |event           |-3.0     |
|org_hv3a3zmf|Gamma Data 2   |Media        |eu-

In [ ]:
customers_raw.printSchema()


root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: string (nullable = true)



In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, BooleanType,
    DateType, DoubleType
)

customers_schema = StructType([
    StructField("org_id", StringType(), True),
    StructField("org_name", StringType(), True),
    StructField("industry", StringType(), True),
    StructField("hq_region", StringType(), True),
    StructField("plan_tier", StringType(), True),
    StructField("is_enterprise", BooleanType(), True),
    StructField("signup_date", DateType(), True),
    StructField("sales_rep", StringType(), True),
    StructField("lifecycle_stage", StringType(), True),
    StructField("marketing_source", StringType(), True),
    StructField("nps_score", DoubleType(), True)
])

print(customers_schema.simpleString())


struct<org_id:string,org_name:string,industry:string,hq_region:string,plan_tier:string,is_enterprise:boolean,signup_date:date,sales_rep:string,lifecycle_stage:string,marketing_source:string,nps_score:double>


In [ ]:
customers_bronze = (
    spark.read
    .option("header", True)
    .option("dateFormat", "yyyy-MM-dd")
    .schema(customers_schema)
    .csv(str(LANDING_PATH / "customers_orgs.csv"))
)

customers_bronze.printSchema()
customers_bronze.show(5, truncate=False)

print("Cantidad de registros:", customers_bronze.count())


root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)

+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_id      |org_name       |industry     |hq_region |plan_tier|is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|
+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_xaji0y6d|Nimbus Labs 0  |Education    |sa-east   |standard |true         |2025-05-26 |rep_

In [ ]:
from pyspark.sql.functions import current_timestamp, input_file_name

customers_bronze = (
    customers_bronze
    .withColumn("ingest_ts", current_timestamp())
    .withColumn("source_file", input_file_name())
)

customers_bronze.select(
    "org_id",
    "org_name",
    "signup_date",
    "nps_score",
    "ingest_ts",
    "source_file"
).show(5, truncate=False)


+------------+---------------+-----------+---------+--------------------------+--------------------------------------------------------------------------------------+
|org_id      |org_name       |signup_date|nps_score|ingest_ts                 |source_file                                                                           |
+------------+---------------+-----------+---------+--------------------------+--------------------------------------------------------------------------------------+
|org_xaji0y6d|Nimbus Labs 0  |2025-05-26 |NULL     |2026-11-12 20:24:29.637732|file:///content/drive/MyDrive/cloud-provider-analytics-data/landing/customers_orgs.csv|
|org_pbhsahxt|Nova Tech 1    |2025-06-16 |-3.0     |2026-11-12 20:24:29.637732|file:///content/drive/MyDrive/cloud-provider-analytics-data/landing/customers_orgs.csv|
|org_hv3a3zmf|Gamma Data 2   |2025-05-12 |4.0      |2026-11-12 20:24:29.637732|file:///content/drive/MyDrive/cloud-provider-analytics-data/landing/customers_orgs.csv

In [ ]:
from pyspark.sql import functions as F

print("Total de registros:", customers_bronze.count())

print(
    "org_id duplicados:",
    customers_bronze.count()
    - customers_bronze.select("org_id").distinct().count()
)

print(
    "Fechas nulas:",
    customers_bronze.filter(F.col("signup_date").isNull()).count()
)

print(
    "NPS nulos:",
    customers_bronze.filter(F.col("nps_score").isNull()).count()
)

print(
    "NPS fuera de rango:",
    customers_bronze.filter(
        (F.col("nps_score") < -100) |
        (F.col("nps_score") > 100)
    ).count()
)


Total de registros: 80
org_id duplicados: 0
Fechas nulas: 0
NPS nulos: 11
NPS fuera de rango: 1


In [ ]:
ruta_customers_bronze = BRONZE_PATH / "customers_orgs"

(
    customers_bronze
    .write
    .mode("overwrite")
    .parquet(str(ruta_customers_bronze))
)

print("Archivo Bronze generado en:")
print(ruta_customers_bronze)


Archivo Bronze generado en:
/content/drive/MyDrive/cloud-provider-analytics-data/bronze/customers_orgs


In [ ]:
customers_verificacion = spark.read.parquet(
    str(ruta_customers_bronze)
)

print("Registros guardados:", customers_verificacion.count())

customers_verificacion.printSchema()

customers_verificacion.select(
    "org_id",
    "org_name",
    "nps_score",
    "ingest_ts",
    "source_file"
).show(5, truncate=False)


Registros guardados: 80
root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

+------------+---------------+---------+--------------------------+--------------------------------------------------------------------------------------+
|org_id      |org_name       |nps_score|ingest_ts                 |source_file                                                                           |
+------------+---------------+---------+--------------------------+------------------------------------------

In [ ]:
# Leemos users.csv sin inferir los tipos de datos
users_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(str(LANDING_PATH / "users.csv"))
)

print("Cantidad de registros:", users_raw.count())

print("\nColumnas disponibles:")
print(users_raw.columns)

print("\nEsquema original:")
users_raw.printSchema()

print("\nPrimeros registros:")
users_raw.show(5, truncate=False)


Cantidad de registros: 800

Columnas disponibles:
['user_id', 'org_id', 'email', 'role', 'active', 'created_at', 'last_login']

Esquema original:
root
 |-- user_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- role: string (nullable = true)
 |-- active: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- last_login: string (nullable = true)


Primeros registros:
+-------------+------------+-------------------------+-----------+------+----------+----------+
|user_id      |org_id      |email                    |role       |active|created_at|last_login|
+-------------+------------+-------------------------+-----------+------+----------+----------+
|user_9tze5r5u|org_3t60rjiw|user_9tze5r5u@example.com|devops     |True  |2025-07-10|NULL      |
|user_qpgb7r3o|org_afeyuhz1|user_qpgb7r3o@example.com|developer  |True  |2025-07-15|2025-08-10|
|user_cwbfuk9e|org_ykl1cq99|user_cwbfuk9e@example.com|developer  |True  |20

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    DateType
)

users_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("email", StringType(), True),
    StructField("role", StringType(), True),
    StructField("active", BooleanType(), True),
    StructField("created_at", DateType(), True),
    StructField("last_login", DateType(), True)
])

print(users_schema.simpleString())


struct<user_id:string,org_id:string,email:string,role:string,active:boolean,created_at:date,last_login:date>


In [ ]:
users_bronze = (
    spark.read
    .option("header", True)
    .option("dateFormat", "yyyy-MM-dd")
    .schema(users_schema)
    .csv(str(LANDING_PATH / "users.csv"))
)

print("Esquema de users:")
users_bronze.printSchema()

print("\nPrimeros registros:")
users_bronze.show(5, truncate=False)

print("Cantidad de registros:", users_bronze.count())


Esquema de users:
root
 |-- user_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- role: string (nullable = true)
 |-- active: boolean (nullable = true)
 |-- created_at: date (nullable = true)
 |-- last_login: date (nullable = true)


Primeros registros:
+-------------+------------+-------------------------+-----------+------+----------+----------+
|user_id      |org_id      |email                    |role       |active|created_at|last_login|
+-------------+------------+-------------------------+-----------+------+----------+----------+
|user_9tze5r5u|org_3t60rjiw|user_9tze5r5u@example.com|devops     |true  |2025-07-10|NULL      |
|user_qpgb7r3o|org_afeyuhz1|user_qpgb7r3o@example.com|developer  |true  |2025-07-15|2025-08-10|
|user_cwbfuk9e|org_ykl1cq99|user_cwbfuk9e@example.com|developer  |true  |2025-06-12|2025-07-26|
|user_1v2isqp4|org_g8sbi4q2|user_1v2isqp4@example.com|developer  |false |2025-05-12|NULL      |
|user_9kwv08hh

In [ ]:
from pyspark.sql.functions import current_timestamp, input_file_name

users_bronze = (
    users_bronze
    .withColumn("ingest_ts", current_timestamp())
    .withColumn("source_file", input_file_name())
)

users_bronze.select(
    "user_id",
    "org_id",
    "active",
    "created_at",
    "last_login",
    "ingest_ts",
    "source_file"
).show(5, truncate=False)


+-------------+------------+------+----------+----------+--------------------------+-----------------------------------------------------------------------------+
|user_id      |org_id      |active|created_at|last_login|ingest_ts                 |source_file                                                                  |
+-------------+------------+------+----------+----------+--------------------------+-----------------------------------------------------------------------------+
|user_9tze5r5u|org_3t60rjiw|true  |2025-07-10|NULL      |2026-11-12 20:26:17.893116|file:///content/drive/MyDrive/cloud-provider-analytics-data/landing/users.csv|
|user_qpgb7r3o|org_afeyuhz1|true  |2025-07-15|2025-08-10|2026-11-12 20:26:17.893116|file:///content/drive/MyDrive/cloud-provider-analytics-data/landing/users.csv|
|user_cwbfuk9e|org_ykl1cq99|true  |2025-06-12|2025-07-26|2026-11-12 20:26:17.893116|file:///content/drive/MyDrive/cloud-provider-analytics-data/landing/users.csv|
|user_1v2isqp4|org_g8s

In [ ]:
from pyspark.sql import functions as F

print("Total de registros:", users_bronze.count())

print(
    "user_id duplicados:",
    users_bronze.count()
    - users_bronze.select("user_id").distinct().count()
)

print(
    "user_id nulos:",
    users_bronze.filter(F.col("user_id").isNull()).count()
)

print(
    "org_id nulos:",
    users_bronze.filter(F.col("org_id").isNull()).count()
)

print(
    "created_at nulos:",
    users_bronze.filter(F.col("created_at").isNull()).count()
)

print(
    "last_login nulos:",
    users_bronze.filter(F.col("last_login").isNull()).count()
)


Total de registros: 800
user_id duplicados: 0
user_id nulos: 0
org_id nulos: 0
created_at nulos: 0
last_login nulos: 139


In [ ]:
ruta_users_bronze = BRONZE_PATH / "users"

(
    users_bronze
    .write
    .mode("overwrite")
    .parquet(str(ruta_users_bronze))
)

print("Datos de users guardados en:")
print(ruta_users_bronze)


Datos de users guardados en:
/content/drive/MyDrive/cloud-provider-analytics-data/bronze/users


In [ ]:
users_verificacion = spark.read.parquet(
    str(ruta_users_bronze)
)

print(
    "Registros guardados:",
    users_verificacion.count()
)

users_verificacion.printSchema()

users_verificacion.select(
    "user_id",
    "org_id",
    "active",
    "created_at",
    "last_login",
    "ingest_ts",
    "source_file"
).show(5, truncate=False)


Registros guardados: 800
root
 |-- user_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- role: string (nullable = true)
 |-- active: boolean (nullable = true)
 |-- created_at: date (nullable = true)
 |-- last_login: date (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

+-------------+------------+------+----------+----------+--------------------------+-----------------------------------------------------------------------------+
|user_id      |org_id      |active|created_at|last_login|ingest_ts                 |source_file                                                                  |
+-------------+------------+------+----------+----------+--------------------------+-----------------------------------------------------------------------------+
|user_9tze5r5u|org_3t60rjiw|true  |2025-07-10|NULL      |2026-11-12 20:27:01.119925|file:///content/drive/MyDrive/cloud-prov

In [ ]:
# Leemos billing_monthly.csv sin inferir tipos
billing_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(str(LANDING_PATH / "billing_monthly.csv"))
)

print("Cantidad de registros:", billing_raw.count())

print("\nColumnas disponibles:")
print(billing_raw.columns)

print("\nEsquema original:")
billing_raw.printSchema()

print("\nPrimeros registros:")
billing_raw.show(5, truncate=False)


Cantidad de registros: 240

Columnas disponibles:
['invoice_id', 'org_id', 'month', 'subtotal', 'credits', 'taxes', 'currency', 'exchange_rate_to_usd']

Esquema original:
root
 |-- invoice_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- month: string (nullable = true)
 |-- subtotal: string (nullable = true)
 |-- credits: string (nullable = true)
 |-- taxes: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- exchange_rate_to_usd: string (nullable = true)


Primeros registros:
+------------+------------+----------+--------+-------+------+--------+--------------------+
|invoice_id  |org_id      |month     |subtotal|credits|taxes |currency|exchange_rate_to_usd|
+------------+------------+----------+--------+-------+------+--------+--------------------+
|inv_nym31sk0|org_xaji0y6d|2025-06-01|1133.34 |NULL   |238.0 |USD     |0.97684             |
|inv_540sejok|org_xaji0y6d|2025-07-01|619.17  |NULL   |130.03|USD     |1.00331             |
|inv_ilp4vn2

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    DecimalType
)

billing_schema = StructType([
    StructField("invoice_id", StringType(), True),
    StructField("org_id", StringType(), True),
    StructField("month", DateType(), True),
    StructField("subtotal", DecimalType(18, 2), True),
    StructField("credits", DecimalType(18, 2), True),
    StructField("taxes", DecimalType(18, 2), True),
    StructField("currency", StringType(), True),
    StructField("exchange_rate_to_usd", DecimalType(18, 8), True)
])

print(billing_schema.simpleString())


struct<invoice_id:string,org_id:string,month:date,subtotal:decimal(18,2),credits:decimal(18,2),taxes:decimal(18,2),currency:string,exchange_rate_to_usd:decimal(18,8)>


In [ ]:
billing_bronze = (
    spark.read
    .option("header", True)
    .option("dateFormat", "yyyy-MM-dd")
    .schema(billing_schema)
    .csv(str(LANDING_PATH / "billing_monthly.csv"))
)

print("Esquema de billing:")
billing_bronze.printSchema()

print("\nPrimeros registros:")
billing_bronze.show(5, truncate=False)

print("Cantidad de registros:", billing_bronze.count())


Esquema de billing:
root
 |-- invoice_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- month: date (nullable = true)
 |-- subtotal: decimal(18,2) (nullable = true)
 |-- credits: decimal(18,2) (nullable = true)
 |-- taxes: decimal(18,2) (nullable = true)
 |-- currency: string (nullable = true)
 |-- exchange_rate_to_usd: decimal(18,8) (nullable = true)


Primeros registros:
+------------+------------+----------+--------+-------+------+--------+--------------------+
|invoice_id  |org_id      |month     |subtotal|credits|taxes |currency|exchange_rate_to_usd|
+------------+------------+----------+--------+-------+------+--------+--------------------+
|inv_nym31sk0|org_xaji0y6d|2025-06-01|1133.34 |NULL   |238.00|USD     |0.97684000          |
|inv_540sejok|org_xaji0y6d|2025-07-01|619.17  |NULL   |130.03|USD     |1.00331000          |
|inv_ilp4vn2v|org_xaji0y6d|2025-08-01|540.79  |NULL   |113.57|ARS     |0.00160000          |
|inv_27nls4sz|org_pbhsahxt|2025-06-01|520.91

In [ ]:
from pyspark.sql.functions import (
    current_timestamp,
    input_file_name
)

billing_bronze = (
    billing_bronze
    .withColumn("ingest_ts", current_timestamp())
    .withColumn("source_file", input_file_name())
)

billing_bronze.select(
    "invoice_id",
    "org_id",
    "month",
    "subtotal",
    "credits",
    "taxes",
    "currency",
    "exchange_rate_to_usd",
    "ingest_ts",
    "source_file"
).show(5, truncate=False)


+------------+------------+----------+--------+-------+------+--------+--------------------+--------------------------+---------------------------------------------------------------------------------------+
|invoice_id  |org_id      |month     |subtotal|credits|taxes |currency|exchange_rate_to_usd|ingest_ts                 |source_file                                                                            |
+------------+------------+----------+--------+-------+------+--------+--------------------+--------------------------+---------------------------------------------------------------------------------------+
|inv_nym31sk0|org_xaji0y6d|2025-06-01|1133.34 |NULL   |238.00|USD     |0.97684000          |2026-11-12 20:28:57.448984|file:///content/drive/MyDrive/cloud-provider-analytics-data/landing/billing_monthly.csv|
|inv_540sejok|org_xaji0y6d|2025-07-01|619.17  |NULL   |130.03|USD     |1.00331000          |2026-11-12 20:28:57.448984|file:///content/drive/MyDrive/cloud-provider-anal

In [ ]:
from pyspark.sql import functions as F

print("Total de registros:", billing_bronze.count())

print(
    "invoice_id duplicados:",
    billing_bronze.count()
    - billing_bronze.select("invoice_id").distinct().count()
)

print(
    "invoice_id nulos:",
    billing_bronze.filter(
        F.col("invoice_id").isNull()
    ).count()
)

print(
    "org_id nulos:",
    billing_bronze.filter(
        F.col("org_id").isNull()
    ).count()
)

print(
    "month nulos:",
    billing_bronze.filter(
        F.col("month").isNull()
    ).count()
)

print(
    "credits nulos:",
    billing_bronze.filter(
        F.col("credits").isNull()
    ).count()
)

print(
    "exchange_rate_to_usd nulos:",
    billing_bronze.filter(
        F.col("exchange_rate_to_usd").isNull()
    ).count()
)


Total de registros: 240
invoice_id duplicados: 0
invoice_id nulos: 0
org_id nulos: 0
month nulos: 0
credits nulos: 137
exchange_rate_to_usd nulos: 0


In [ ]:
ruta_billing_bronze = BRONZE_PATH / "billing_monthly"

(
    billing_bronze
    .write
    .mode("overwrite")
    .parquet(str(ruta_billing_bronze))
)

print("Datos de billing guardados en:")
print(ruta_billing_bronze)


Datos de billing guardados en:
/content/drive/MyDrive/cloud-provider-analytics-data/bronze/billing_monthly


In [ ]:
billing_verificacion = spark.read.parquet(
    str(ruta_billing_bronze)
)

print(
    "Registros guardados:",
    billing_verificacion.count()
)

billing_verificacion.printSchema()

billing_verificacion.select(
    "invoice_id",
    "org_id",
    "month",
    "subtotal",
    "credits",
    "taxes",
    "currency",
    "exchange_rate_to_usd",
    "ingest_ts",
    "source_file"
).show(5, truncate=False)


Registros guardados: 240
root
 |-- invoice_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- month: date (nullable = true)
 |-- subtotal: decimal(18,2) (nullable = true)
 |-- credits: decimal(18,2) (nullable = true)
 |-- taxes: decimal(18,2) (nullable = true)
 |-- currency: string (nullable = true)
 |-- exchange_rate_to_usd: decimal(18,8) (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

+------------+------------+----------+--------+-------+------+--------+--------------------+--------------------------+---------------------------------------------------------------------------------------+
|invoice_id  |org_id      |month     |subtotal|credits|taxes |currency|exchange_rate_to_usd|ingest_ts                 |source_file                                                                            |
+------------+------------+----------+--------+-------+------+--------+--------------------+-----------------------